#### 7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which Unity Catalog groups should have access, and how you'd audit access after the fact.


## Governance Model for Cyntexa

### 1. PII Classification & Sensitive Data Inventory

#### Customer/User Tables
| Table | Sensitive Columns | PII Type | Classification |
|-------|------------------|----------|----------------|
| `customers` | `email`, `phone`, `ssn`, `date_of_birth` | Direct Identifiers | HIGH |
| `customers` | `first_name`, `last_name`, `address`, `city`, `state`, `zip_code` | Personal Info | MEDIUM |
| `customer_profiles` | `credit_card_number`, `bank_account` | Financial | CRITICAL |
| `customer_preferences` | `ip_address`, `device_id` | Pseudonymous | MEDIUM |

#### Employee Tables
| Table | Sensitive Columns | PII Type | Classification |
|-------|------------------|----------|----------------|
| `employees` | `ssn`, `tax_id`, `salary`, `bank_details` | Financial/ID | CRITICAL |
| `employees` | `email`, `phone`, `home_address`, `emergency_contact` | Contact Info | HIGH |
| `hr_records` | `performance_reviews`, `disciplinary_actions`, `health_records` | Sensitive HR | CRITICAL |

#### Transaction Tables
| Table | Sensitive Columns | PII Type | Classification |
|-------|------------------|----------|----------------|
| `orders` | `customer_id`, `billing_address`, `shipping_address` | Linked PII | MEDIUM |
| `payments` | `credit_card_last4`, `payment_method`, `transaction_amount` | Financial | HIGH |
| `transactions` | `user_id`, `ip_address`, `geolocation` | Behavioral | MEDIUM |

### 2. Unity Catalog Access Control Model

#### Recommended Group Structure

```
├── data_governance_admins (Metastore admins)
├── security_compliance_team (Audit access)
├── data_engineering_team (Full access to raw/bronze)
├── analytics_team (Masked/aggregated access)
├── data_science_team (Pseudonymized data access)
├── business_users (Dashboard/report access only)
├── customer_support (Limited PII for support cases)
├── finance_team (Financial data access)
└── hr_team (Employee data access)
```

#### Access Matrix

| Group | Raw PII Access | Masked Data | Aggregated | Audit Logs |
|-------|----------------|-------------|------------|------------|
| `data_governance_admins` | ✓ | ✓ | ✓ | ✓ |
| `security_compliance_team` | ✗ | ✓ | ✓ | ✓ |
| `data_engineering_team` | ✓ (Limited) | ✓ | ✓ | ✗ |
| `analytics_team` | ✗ | ✓ | ✓ | ✗ |
| `data_science_team` | ✗ | ✓ | ✓ | ✗ |
| `business_users` | ✗ | ✗ | ✓ | ✗ |
| `customer_support` | ✓ (ABAC-filtered) | ✓ | ✗ | ✗ |
| `finance_team` | ✓ (Finance cols only) | ✓ | ✓ | ✗ |
| `hr_team` | ✓ (Employee data only) | ✓ | ✓ | ✗ |

### 3. Implementation Strategy

#### A. Row-Level Security (Row Filters)
```sql
-- Customer support: only access customers in their assigned region
CREATE FUNCTION customer_support_filter(region STRING)
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('customer_support'), 
          region = current_user_region(), 
          TRUE);

ALTER TABLE customers SET ROW FILTER customer_support_filter ON (state);
```

#### B. Column Masking
```sql
-- Mask email for analytics team
CREATE FUNCTION mask_email(email STRING)
RETURN CASE
  WHEN IS_ACCOUNT_GROUP_MEMBER('data_governance_admins') THEN email
  WHEN IS_ACCOUNT_GROUP_MEMBER('customer_support') THEN email
  ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
END;

ALTER TABLE customers ALTER COLUMN email SET MASK mask_email;

-- Mask SSN completely
CREATE FUNCTION mask_ssn(ssn STRING)
RETURN CASE
  WHEN IS_ACCOUNT_GROUP_MEMBER('data_governance_admins') THEN ssn
  WHEN IS_ACCOUNT_GROUP_MEMBER('hr_team') THEN ssn
  ELSE '***-**-' || RIGHT(ssn, 4)
END;

ALTER TABLE employees ALTER COLUMN ssn SET MASK mask_ssn;
```

#### C. Data Classification Tags
```sql
-- Tag sensitive columns
ALTER TABLE customers ALTER COLUMN email SET TAGS ('pii' = 'high', 'gdpr' = 'article_9');
ALTER TABLE customers ALTER COLUMN ssn SET TAGS ('pii' = 'critical', 'compliance' = 'pci_dss');
ALTER TABLE payments ALTER COLUMN credit_card_number SET TAGS ('pii' = 'critical', 'compliance' = 'pci_dss');
```

### 4. Audit Strategy

#### A. System Tables for Access Monitoring
```sql
-- Query 1: Track who accessed PII columns
SELECT 
  event_time,
  user_identity.email as user_email,
  request_params.table_full_name,
  request_params.columns_accessed,
  action_name
FROM system.access.audit
WHERE 
  action_name = 'getTable'
  AND event_date >= CURRENT_DATE - INTERVAL 30 DAYS
  AND array_contains(request_params.columns_accessed, 'ssn')
ORDER BY event_time DESC;
```

```sql
-- Query 2: Monitor failed access attempts
SELECT 
  event_time,
  user_identity.email,
  request_params.table_full_name,
  response.status_code,
  response.error_message
FROM system.access.audit
WHERE 
  response.status_code IN (401, 403)
  AND event_date >= CURRENT_DATE - INTERVAL 7 DAYS
GROUP BY ALL
ORDER BY event_time DESC;
```

```sql
-- Query 3: High-volume PII access anomaly detection
WITH access_counts AS (
  SELECT 
    user_identity.email as user_email,
    DATE(event_time) as access_date,
    COUNT(*) as access_count,
    COUNT(DISTINCT request_params.table_full_name) as distinct_tables
  FROM system.access.audit
  WHERE 
    event_date >= CURRENT_DATE - INTERVAL 7 DAYS
    AND action_name IN ('getTable', 'readTable')
  GROUP BY 1, 2
)
SELECT *
FROM access_counts
WHERE access_count > 1000  -- Flag unusual high-volume access
ORDER BY access_count DESC;
```

#### B. Data Lineage Tracking
```sql
-- Query 4: Trace downstream usage of PII tables
SELECT 
  source_table_full_name,
  target_table_full_name,
  entity_type,
  created_by,
  created_at
FROM system.access.table_lineage
WHERE source_table_full_name LIKE '%customers%'
ORDER BY created_at DESC;
```

#### C. Scheduled Compliance Reports
```sql
-- Query 5: Weekly PII access summary by group
SELECT 
  DATE_TRUNC('week', event_time) as week,
  user_identity.email,
  request_params.table_full_name,
  COUNT(*) as access_count,
  COUNT(DISTINCT DATE(event_time)) as days_accessed
FROM system.access.audit
WHERE 
  event_date >= CURRENT_DATE - INTERVAL 90 DAYS
  AND request_params.table_full_name IN (
    'catalog.schema.customers',
    'catalog.schema.employees',
    'catalog.schema.payments'
  )
GROUP BY ALL
HAVING access_count > 10
ORDER BY week DESC, access_count DESC;
```

### 5. Ongoing Governance Procedures

#### Monthly Reviews
- Review `system.access.audit` for unusual access patterns
- Validate group memberships against HR records
- Check for new tables/columns requiring classification
- Review masked column functions for effectiveness

#### Quarterly Audits
- Full data catalog scan for untagged sensitive columns
- Access recertification for all groups
- Test row filters and column masks
- Update compliance mappings (GDPR, CCPA, HIPAA)

#### Automated Alerts
```sql
-- Set up alerts for:
-- 1. First-time access to CRITICAL PII by new users
-- 2. After-hours access to sensitive tables
-- 3. Failed permission attempts > threshold
-- 4. Export/download of large PII datasets
-- 5. Changes to security policies or permissions
```

### 6. Documentation & Training
- Maintain a **Data Dictionary** with PII classifications
- Require annual **data privacy training** for all users
- Document **break-glass procedures** for emergency PII access
- Create **runbooks** for access request workflows
- Establish a **Data Governance Council** for policy decisions


#### 8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the edge case of a customer record that hasn't changed since the last load (it should not create a false new version).

In [0]:
%sql
CREATE OR REPLACE TABLE source_customers (
  customer_id INT,
  email STRING,
  phone STRING,
  address STRING,
  city STRING,
  state STRING,
  status STRING
);

INSERT INTO source_customers VALUES
  (1, 'john@example.com', '555-1234', '123 Main St', 'Springfield', 'IL', 'Active'),
  (2, 'jane@example.com', '555-5678', '456 Oak Ave', 'Chicago', 'IL', 'Active'),
  (3, 'bob@example.com', '555-9012', '789 Pine Rd', 'Denver', 'CO', 'Active');

CREATE OR REPLACE TABLE customer_history (
  customer_id INT,
  email STRING,
  phone STRING,
  address STRING,
  city STRING,
  state STRING,
  status STRING,
  effective_date TIMESTAMP,
  end_date TIMESTAMP,
  is_current BOOLEAN
);

CREATE OR REPLACE TEMPORARY VIEW current_customers AS
SELECT 
  customer_id,
  email,
  phone,
  address,
  city,
  state,
  status,
  CURRENT_TIMESTAMP() as effective_date,
  NULL as end_date,
  TRUE as is_current
FROM source_customers;

MERGE INTO customer_history target
USING (
  SELECT 
    c.customer_id,
    c.email,
    c.phone,
    c.address,
    c.city,
    c.state,
    c.status,
    c.effective_date,
    c.end_date,
    c.is_current,
    h.email as old_email,
    h.phone as old_phone,
    h.address as old_address,
    h.city as old_city,
    h.state as old_state,
    h.status as old_status
  FROM current_customers c
  LEFT JOIN customer_history h
    ON c.customer_id = h.customer_id
    AND h.is_current = TRUE
) source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND (
  COALESCE(source.email, '') != COALESCE(source.old_email, '') OR
  COALESCE(source.phone, '') != COALESCE(source.old_phone, '') OR
  COALESCE(source.address, '') != COALESCE(source.old_address, '') OR
  COALESCE(source.city, '') != COALESCE(source.old_city, '') OR
  COALESCE(source.state, '') != COALESCE(source.old_state, '') OR
  COALESCE(source.status, '') != COALESCE(source.old_status, '')
)
THEN UPDATE SET
  target.is_current = FALSE,
  target.end_date = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    email,
    phone,
    address,
    city,
    state,
    status,
    effective_date,
    end_date,
    is_current
  )
  VALUES (
    source.customer_id,
    source.email,
    source.phone,
    source.address,
    source.city,
    source.state,
    source.status,
    source.effective_date,
    source.end_date,
    source.is_current
  );

INSERT INTO customer_history
SELECT 
  c.customer_id,
  c.email,
  c.phone,
  c.address,
  c.city,
  c.state,
  c.status,
  CURRENT_TIMESTAMP() as effective_date,
  NULL as end_date,
  TRUE as is_current
FROM current_customers c
INNER JOIN customer_history h
  ON c.customer_id = h.customer_id
  AND h.end_date = CURRENT_TIMESTAMP()
WHERE (
  COALESCE(c.email, '') != COALESCE(h.email, '') OR
  COALESCE(c.phone, '') != COALESCE(h.phone, '') OR
  COALESCE(c.address, '') != COALESCE(h.address, '') OR
  COALESCE(c.city, '') != COALESCE(h.city, '') OR
  COALESCE(c.state, '') != COALESCE(h.state, '') OR
  COALESCE(c.status, '') != COALESCE(h.status, '')
);


#### 9. (Data Analyst) Using the SCD Type 2 history table, build a customer retention/churn-over-time report that depends on point-in-time correctness, and explain why a simple 'current state' table would give the wrong answer here.

In [0]:
%sql
WITH monthly_spine AS (
  SELECT 
    EXPLODE(SEQUENCE(
      DATE_TRUNC('month', (SELECT MIN(effective_date) FROM customer_history)),
      DATE_TRUNC('month', CURRENT_DATE()),
      INTERVAL 1 MONTH
    )) AS month_start
),

active_customers_by_month AS (
  SELECT 
    m.month_start,
    h.customer_id,
    h.status,
    h.effective_date,
    h.end_date
  FROM monthly_spine m
  CROSS JOIN customer_history h
  WHERE h.effective_date <= LAST_DAY(m.month_start)
    AND (h.end_date IS NULL OR h.end_date > m.month_start)
    AND h.status = 'Active'
),

monthly_active_counts AS (
  SELECT 
    month_start,
    COUNT(DISTINCT customer_id) AS active_customers
  FROM active_customers_by_month
  GROUP BY month_start
),

churn_events AS (
  SELECT 
    DATE_TRUNC('month', effective_date) AS churn_month,
    customer_id
  FROM customer_history
  WHERE status != 'Active'
    AND customer_id IN (
      SELECT customer_id 
      FROM customer_history 
      WHERE status = 'Active'
    )
),

monthly_churn_counts AS (
  SELECT 
    churn_month AS month_start,
    COUNT(DISTINCT customer_id) AS churned_customers
  FROM churn_events
  GROUP BY churn_month
),

retention_metrics AS (
  SELECT 
    a.month_start,
    a.active_customers,
    COALESCE(c.churned_customers, 0) AS churned_customers,
    a.active_customers - COALESCE(LAG(a.active_customers) OVER (ORDER BY a.month_start), 0) AS net_change,
    ROUND(100.0 * COALESCE(c.churned_customers, 0) / NULLIF(a.active_customers, 0), 2) AS churn_rate_pct,
    ROUND(100.0 * (1 - COALESCE(c.churned_customers, 0) / NULLIF(a.active_customers, 0)), 2) AS retention_rate_pct
  FROM monthly_active_counts a
  LEFT JOIN monthly_churn_counts c ON a.month_start = c.month_start
)

SELECT 
  DATE_FORMAT(month_start, 'yyyy-MM') AS month,
  active_customers,
  churned_customers,
  net_change,
  churn_rate_pct,
  retention_rate_pct
FROM retention_metrics
ORDER BY month_start;

SELECT '=== WHY CURRENT STATE TABLE FAILS ===' AS explanation;

WITH current_state_wrong AS (
  SELECT 
    COUNT(*) AS total_customers,
    SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_now,
    SUM(CASE WHEN status != 'Active' THEN 1 ELSE 0 END) AS inactive_now
  FROM customer_history
  WHERE is_current = TRUE
)
SELECT 
  'Current state only shows TODAY snapshot' AS problem,
  total_customers,
  active_now,
  inactive_now,
  'Cannot tell WHEN customers churned' AS limitation_1,
  'Cannot calculate monthly retention trends' AS limitation_2,
  'Cannot see historical active counts' AS limitation_3,
  'Loses all temporal context for analysis' AS limitation_4
FROM current_state_wrong;

SELECT '=== EXAMPLE: Point-in-Time Correctness ===' AS explanation;

SELECT 
  customer_id,
  status,
  effective_date,
  end_date,
  is_current,
  'Shows full history of status changes' AS why_this_matters
FROM customer_history
WHERE customer_id IN (1, 2)
ORDER BY customer_id, effective_date;